# 读取 Parquet 数据集并分析字段缺失程度与分布

本 Notebook 用于对完整的 Parquet 数据集进行处理与分析：
- **源文件**: `C:\Users\小鸟giaogiao叫\Desktop\近期文档\yield_flat_table_joined_100.parquet`
- **主要分析与清洗流程**:
  - 全量读取 Parquet 数据集（不限制行数）。
  - 查看各字段的缺失值统计与百分比分布。
  - 剔除缺失率前 8 的列（后续分析中不予考虑）。
  - 删除 `rfh1wc-first` 与 `rfppwc` 缺失的行记录。
  - 查看清洗后数据中 `group` 字段的分布比例。

In [ ]:
import os
import pandas as pd
import numpy as np

# 定义文件路径（使用 raw string 避免 Windows 路径中反斜杠的转义问题）
parquet_path = r"C://Users//uif77331//Desktop//111//data_clean//yield_flat_table_joined_100.parquet"
print(f"源 Parquet 文件路径: {parquet_path}")

: 

## 读取完整 Parquet 数据集

直接使用 Pandas 全量读取 Parquet 文件。

In [ ]:
df = None

try:
    print("正在全量读取 Parquet 数据集...")
    df = pd.read_parquet(parquet_path)
    print(f"读取成功！数据集共计: {df.shape[0]:,} 行，{df.shape[1]} 列。")
except Exception as e:
    print(f"读取失败: {e}")
    raise e

# 展示数据前几行
if df is not None:
    display(df.head())


## 数据清洗流程：查看每个字段的缺失程度并剔除/过滤

对全量数据进行缺失值（Null/NaN）统计，并进行如下清洗操作：
1. 计算所有字段的缺失分布。
2. 剔除缺失率前 8 的列。
3. 删除 `rfh1wc-first` 与 `rfppwc` 缺失的行记录。

In [ ]:
if df is not None:
    # 1. 计算每个字段的缺失值数量与缺失比例
    missing_count = df.isnull().sum()
    missing_ratio = df.isnull().mean() * 100
    
    # 2. 整合为统计结果表并按缺失比例降序排列
    missing_df = pd.DataFrame({
        '字段名称 (Column)': df.columns,
        '缺失值数量 (Missing Count)': missing_count.values,
        '缺失比例 (Missing Ratio %)': missing_ratio.round(4).values
    }).sort_values(by='缺失比例 (Missing Ratio %)', ascending=False).reset_index(drop=True)
    
    # 3. 基础宏观指标输出
    total_rows = len(df)
    missing_cols_count = len(missing_df[missing_df['缺失值数量 (Missing Count)'] > 0])
    print(f"📊 全量数据缺失统计：")
    print(f"   - 数据总样本行数: {total_rows:,} 行")
    print(f"   - 存在缺失值的字段数: {missing_cols_count} / {len(df.columns)} 个")
    
    # 展示排名前 50 的字段缺失情况
    print("\n--- 字段缺失程度一览表 (降序) ---")
    display(missing_df.head(50))
    
    # 4. 可视化分析缺失值比例分布（仅展示有缺失的字段）
    has_missing = missing_df[missing_df['缺失值数量 (Missing Count)'] > 0]
    if not has_missing.empty:
        try:
            import matplotlib.pyplot as plt
            import seaborn as sns
            
            plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
            plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
            plt.rcParams['axes.unicode_minus'] = False
            
            # 根据缺失字段个数动态设置图表高度
            plt.figure(figsize=(12, min(10, len(has_missing) * 0.4 + 2)))
            
            sns.barplot(
                x='缺失比例 (Missing Ratio %)', 
                y='字段名称 (Column)', 
                data=has_missing, 
                palette='Reds_r',
                hue='字段名称 (Column)',
                legend=False
            )
            
            plt.title("各字段全量缺失比例分布排行 (%)", fontsize=14, fontweight='bold', pad=15)
            plt.xlabel("缺失比例 (%)", fontsize=12)
            plt.ylabel("字段名称 (Column)", fontsize=12)
            
            # 为每个条形图柱子加标注
            for i, val in enumerate(has_missing['缺失比例 (Missing Ratio %)'].values):
                plt.text(val + 0.1, i, f"{val:.2f}%", va='center', fontsize=9, color='darkred')
                
            plt.tight_layout()
            plt.show()
        except Exception as plot_err:
            print(f"画图失败 (可能是无绘图环境): {plot_err}")
    else:
        print("\n🎉 恭喜！全量数据非常完整，没有任何缺失值！")
        
    # 5. 剔除缺失率前 8 的列
    top_8_missing_cols = missing_df.head(8)['字段名称 (Column)'].tolist()
    print(f"\n🗑️ 剔除缺失率前 8 的列 (不考虑):\n   {top_8_missing_cols}")
    df = df.drop(columns=top_8_missing_cols)
    
    # 6. 删除 rfh1wc-first 与 rfppwc 缺失的行记录
    # 兼容处理中划线与下划线命名
    target_cols = ['rfh1wc_first', 'rfh1wc-first', 'rfppwc']
    cols_to_check = [col for col in df.columns if col.lower() in [c.lower() for c in target_cols]]
    
    if cols_to_check:
        print(f"\n🧹 检查并删除以下列有缺失值 (NaN) 的行: {cols_to_check}")
        before_rows = len(df)
        df = df.dropna(subset=cols_to_check)
        after_rows = len(df)
        print(f"   - 清洗前总行数: {before_rows:,} 行")
        print(f"   - 清洗后总行数: {after_rows:,} 行")
        print(f"   - 已删除缺失行数: {before_rows - after_rows:,} 行")
    else:
        print("\n⚠️ 未在数据中匹配到 'rfh1wc-first' 或 'rfppwc' 字段，跳过行删除步骤。")
        
    print(f"\n✨ 清洗完成！当前数据集 Shape: {df.shape}")
else:
    print("未成功读取到数据，请先运行上方的读取单元格。")

## 查看 `group` 字段下不同值的比例分布

接下来，我们将对清洗后数据中 `group` 字段的非重复值出现的频数以及对应的百分比分布进行统计，并绘制可视化分布图。

In [ ]:
if df is not None:
    # 检查并统一字段名大小写（匹配 'group' 字段）
    group_col = None
    for col in df.columns:
        if col.lower() == 'group':
            group_col = col
            break
            
    if group_col:
        print(f"成功找到字段: '{group_col}'")
        
        # 计算频数和占比（包含缺失值 NaN 的统计）
        counts = df[group_col].value_counts(dropna=False)
        proportions = df[group_col].value_counts(normalize=True, dropna=False) * 100
        
        # 整合为分布明细表
        dist_df = pd.DataFrame({
            '频数 (Count)': counts,
            '比例 (Percentage %)': proportions.round(2)
        })
        
        print("\n--- group 字段值的全量比例分布统计表 ---")
        display(dist_df)
        
        # 绘制比例分布图
        try:
            import matplotlib.pyplot as plt
            import seaborn as sns
            
            # 绘图风格设置
            plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
            plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
            plt.rcParams['axes.unicode_minus'] = False
            
            plt.figure(figsize=(10, 5))
            # 过滤掉空的类别绘制条形图
            plot_data = proportions.dropna()
            # 将索引转换为字符串避免画图报错
            x_labels = [str(x) for x in plot_data.index]
            
            sns.barplot(x=x_labels, y=plot_data.values, palette='viridis', hue=x_labels, legend=False)
            plt.title("group 字段全量取值比例分布图 (%)", fontsize=14, fontweight='bold', pad=15)
            plt.xlabel("group 字段取值", fontsize=12)
            plt.ylabel("比例 (%)", fontsize=12)
            plt.xticks(rotation=30, ha='right')
            
            # 在条形图顶部添加百分比数值标签
            for i, val in enumerate(plot_data.values):
                plt.text(i, val + 0.5, f"{val:.2f}%", ha='center', va='bottom', fontsize=10)
                
            plt.tight_layout()
            plt.show()
        except Exception as plot_err:
            print(f"画图失败 (可能是无绘图环境): {plot_err}")
    else:
        print(f"⚠️ 警告: 未在数据中找到名为 'group' 的字段！当前拥有的字段有:\n{list(df.columns)}")
else:
    print("未成功读取到数据，请先运行上方的读取单元格。")

## 数据转换：将物理指标上限标准值扩大 10 倍

根据要求，我们将 `standard_rfpp` 和 `standard_rfh1` 两个字段的数值均扩大 10 倍，以便与物理指标 `rfppwc_first` 和 `rfh1wc_first` 的量线保持一致。

In [ ]:
if df is not None:
    print("--- 数据转换前数值样例 ---")
    display(df[['group', 'standard_rfpp', 'standard_rfh1']].drop_duplicates().head(10))
    
    # 扩大 10 倍
    df['standard_rfpp'] = df['standard_rfpp'] * 10.0
    df['standard_rfh1'] = df['standard_rfh1'] * 10.0
    
    print("\n✅ 字段 'standard_rfpp' 和 'standard_rfh1' 均已成功扩大 10 倍！")
    print("--- 数据转换后数值样例 ---")
    display(df[['group', 'standard_rfpp', 'standard_rfh1']].drop_duplicates().head(10))
else:
    print("⚠️ 数据未加载，请运行前面的单元格。")

In [ ]:
if df is not None:
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    # 1. 强制转换为数值类型，避免 'str' 与 'float' 比较报错
    df['rfppwc_first'] = pd.to_numeric(df['rfppwc_first'], errors='coerce')
    df['rfh1wc_first'] = pd.to_numeric(df['rfh1wc_first'], errors='coerce')
    df['standard_rfpp'] = pd.to_numeric(df['standard_rfpp'], errors='coerce')
    df['standard_rfh1'] = pd.to_numeric(df['standard_rfh1'], errors='coerce')
    
    # 2. 物理指标异常值判定
    # 超过上限值标记为异常 (即 1，否则为 0)
    df['anomaly_rfpp'] = df['rfppwc_first'] > df['standard_rfpp']
    df['anomaly_rfh1'] = df['rfh1wc_first'] > df['standard_rfh1']
    df['is_anomaly'] = (df['anomaly_rfpp'] | df['anomaly_rfh1']).astype(int)
    
    # 3. 按日度 (ct_shiftdate) 汇总统计正常/异常数以及异常率
    df['date_str'] = pd.to_datetime(df['ct_shiftdate']).dt.strftime('%Y-%m-%d')
    daily_stats = df.groupby('date_str')['is_anomaly'].agg(['count', 'sum', 'mean']).reset_index()
    daily_stats.rename(columns={
        'count': 'total_cnt',
        'sum': 'anomaly_cnt',
        'mean': 'anomaly_rate'
    }, inplace=True)
    daily_stats['normal_cnt'] = daily_stats['total_cnt'] - daily_stats['anomaly_cnt']
    daily_stats['anomaly_rate_pct'] = daily_stats['anomaly_rate'] * 100
    
    # 按照日期升序排列
    daily_stats = daily_stats.sort_values(by='date_str').reset_index(drop=True)
    
    # 展示统计指标明细表
    print("--- 每日物理指标异常统计明细表 ---")
    display(daily_stats[['date_str', 'normal_cnt', 'anomaly_cnt', 'total_cnt', 'anomaly_rate_pct']])
    
    # 4. 绘制双 Y 轴图表
    # 绘图样式与中文字体设置
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False
    
    fig, ax1 = plt.subplots(figsize=(15, 8))
    
    # 4.1 绘制日产量正常/异常堆叠柱状图 (左 Y 轴)
    color_normal = '#1a73e8'   # 经典蓝
    color_anomaly = '#d93025'  # 经典红
    
    dates = daily_stats['date_str']
    ax1.bar(dates, daily_stats['normal_cnt'], color=color_normal, alpha=0.5, label='正常条数', width=0.6)
    ax1.bar(dates, daily_stats['anomaly_cnt'], bottom=daily_stats['normal_cnt'], color=color_anomaly, alpha=0.8, label='异常条数', width=0.6)
    
    ax1.set_xlabel('生产日期 (Date)', labelpad=12, fontsize=12, fontweight='bold')
    ax1.set_ylabel('日度生产条数 / 样本量 (条)', color='black', fontsize=12, fontweight='bold')
    ax1.tick_params(axis='y', labelcolor='black')
    ax1.tick_params(axis='x', rotation=45)
    
    # 4.2 绘制日物理指标异常率曲线 (右 Y 轴)
    ax2 = ax1.twinx()
    ax2.plot(dates, daily_stats['anomaly_rate_pct'], color=color_anomaly, marker='o', linewidth=2.5, label='日异常率 (%)')
    ax2.set_ylabel('物理指标异常率 (%)', color=color_anomaly, fontsize=12, fontweight='bold')
    ax2.tick_params(axis='y', labelcolor=color_anomaly)
    ax2.grid(False)  
    
    # 绘制平均异常率辅助线
    mean_anomaly_rate = daily_stats['anomaly_rate_pct'].mean()
    ax2.axhline(mean_anomaly_rate, color='#f9ab00', linestyle='--', linewidth=1.5, label=f'平均异常率 ({mean_anomaly_rate:.2f}%)')
    
    # 合并图例
    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(handles1 + handles2, labels1 + labels2, loc='upper left', frameon=True, fontsize=10)
    
    # 在折线节点上标注每个日期的具体异常率百分比
    for i, val in enumerate(daily_stats['anomaly_rate_pct']):
        ax2.annotate(f"{val:.2f}%", 
                     xy=(i, val), 
                     xytext=(0, 8), 
                     textcoords="offset points", 
                     ha='center', 
                     va='bottom', 
                     fontsize=9, 
                     weight='bold', 
                     color='darkred')
                     
    plt.title("📊 每日每一个 Barcode 物理指标异常分布与异常率趋势对比", fontsize=15, fontweight='bold', pad=18)
    fig.tight_layout()
    plt.show()
else:
    print("⚠️ 数据未加载，请运行前面的单元格。")


In [ ]:
if df is not None:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    # 1. 确保数值类型正确
    df['rfppwc_first'] = pd.to_numeric(df['rfppwc_first'], errors='coerce')
    df['rfh1wc_first'] = pd.to_numeric(df['rfh1wc_first'], errors='coerce')
    df['standard_rfpp'] = pd.to_numeric(df['standard_rfpp'], errors='coerce')
    df['standard_rfh1'] = pd.to_numeric(df['standard_rfh1'], errors='coerce')
    
    df['date_str'] = pd.to_datetime(df['ct_shiftdate']).dt.strftime('%Y-%m-%d')
    
    # 2. 判定单个 barcode 的异常状态 (超过上限)
    df['anomaly_rfpp'] = df['rfppwc_first'] > df['standard_rfpp']
    df['anomaly_rfh1'] = df['rfh1wc_first'] > df['standard_rfh1']
    df['is_anomaly'] = (df['anomaly_rfpp'] | df['anomaly_rfh1']).astype(int)
    
    # 3. 按日期和组别统计各项指标的异常率
    grouped = df.groupby(['date_str', 'group'])
    anomaly_stats = []
    
    for (date, grp), g_data in grouped:
        if pd.isna(grp):
            continue
        total_count = len(g_data)
        if total_count < 5:
            continue
            
        rate_rfpp = g_data['anomaly_rfpp'].mean() * 100
        rate_rfh1 = g_data['anomaly_rfh1'].mean() * 100
        rate_combined = g_data['is_anomaly'].mean() * 100
        
        anomaly_stats.append({
            'date_str': date,
            'group': grp,
            'rate_rfpp': rate_rfpp,
            'rate_rfh1': rate_rfh1,
            'rate_combined': rate_combined,
            'total_count': total_count
        })
        
    df_anomaly_group = pd.DataFrame(anomaly_stats)
    df_anomaly_group = df_anomaly_group.sort_values(by=['group', 'date_str']).reset_index(drop=True)
    
    # 4. 按 group 绘制各自的单 barcode 异常率折线图
    unique_groups = sorted(df_anomaly_group['group'].unique())
    print(f"检测到以下分组，将依次进行异常率曲线绘制: {unique_groups}")
    
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False
    
    for grp in unique_groups:
        df_grp = df_anomaly_group[df_anomaly_group['group'] == grp].copy()
        if df_grp.empty:
            continue
            
        plt.figure(figsize=(15, 6))
        
        # 绘制三条曲线：综合异常率、rfpp 超标率、rfh1 超标率
        plt.plot(df_grp['date_str'], df_grp['rate_combined'], marker='o', linewidth=2.5, color='#d93025', label='综合异常率 (RFPP或RFH1超标)')
        plt.plot(df_grp['date_str'], df_grp['rate_rfpp'], marker='s', linewidth=1.8, linestyle='--', color='#1a73e8', label='rfppwc_first 超标率')
        plt.plot(df_grp['date_str'], df_grp['rate_rfh1'], marker='^', linewidth=1.8, linestyle='--', color='#f9ab00', label='rfh1wc_first 超标率')
        
        # 在综合折线节点上标注具体异常率
        for i, row in df_grp.reset_index(drop=True).iterrows():
            plt.annotate(f"{row['rate_combined']:.2f}%", 
                         xy=(i, row['rate_combined']), 
                         xytext=(0, 8), 
                         textcoords="offset points", 
                         ha='center', 
                         va='bottom', 
                         fontsize=9, 
                         weight='bold', 
                         color='darkred')
                         
        plt.title(f"📊 分组: {grp} - 单 Barcode 物理指标日度异常率与细分超标率趋势图", fontsize=13, fontweight='bold', pad=15)
        plt.xlabel("生产日期 (Date)", fontsize=11)
        plt.ylabel("异常率 / 超标率 (%)", fontsize=11)
        plt.xticks(rotation=45, ha='right')
        plt.legend(loc='upper right', frameon=True)
        plt.tight_layout()
        plt.show()
else:
    print("⚠️ 数据未加载，请运行前面的单元格。")


In [ ]:
if df is not None:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    # 1. 确保数据类型正确
    df['rfppwc_first'] = pd.to_numeric(df['rfppwc_first'], errors='coerce')
    df['rfh1wc_first'] = pd.to_numeric(df['rfh1wc_first'], errors='coerce')
    df['standard_rfpp'] = pd.to_numeric(df['standard_rfpp'], errors='coerce')
    df['standard_rfh1'] = pd.to_numeric(df['standard_rfh1'], errors='coerce')
    
    df['date_str'] = pd.to_datetime(df['ct_shiftdate']).dt.strftime('%Y-%m-%d')
    
    # 2. 统计每日每个 group 的聚合指标：平均值、标准差、以及 Cpk 指数
    # Cpk(Upper) = (USL - Mean) / (3 * Std)
    def calculate_daily_quality(group_df):
        stats = []
        # 按日期和组进行聚合
        grouped = group_df.groupby(['date_str', 'group'])
        for (date, grp), g_data in grouped:
            if len(g_data) < 5:  # 样本量太少则不计算统计指标
                continue
            
            # 获取该组上限 (standard 在同组内恒定)
            usl_rfpp = g_data['standard_rfpp'].dropna().iloc[0] if not g_data['standard_rfpp'].dropna().empty else None
            usl_rfh1 = g_data['standard_rfh1'].dropna().iloc[0] if not g_data['standard_rfh1'].dropna().empty else None
            
            # 计算 rfpp 的均值和标准差
            mean_rfpp = g_data['rfppwc_first'].mean()
            std_rfpp = g_data['rfppwc_first'].std()
            
            # 计算 rfh1 的均值和标准差
            mean_rfh1 = g_data['rfh1wc_first'].mean()
            std_rfh1 = g_data['rfh1wc_first'].std()
            
            # Cpk 指数计算 (单侧上限能力指数)
            cpk_rfpp = (usl_rfpp - mean_rfpp) / (3 * std_rfpp + 1e-8) if usl_rfpp is not None else np.nan
            cpk_rfh1 = (usl_rfh1 - mean_rfh1) / (3 * std_rfh1 + 1e-8) if usl_rfh1 is not None else np.nan
            
            stats.append({
                'date_str': date,
                'group': grp,
                'sample_size': len(g_data),
                'cpk_rfpp': cpk_rfpp,
                'cpk_rfh1': cpk_rfh1
            })
        return pd.DataFrame(stats)

    daily_quality_df = calculate_daily_quality(df)
    daily_quality_df = daily_quality_df.sort_values(by=['group', 'date_str']).reset_index(drop=True)
    
    # 3. 绘制折线趋势图：将 4 个组的折线绘制在同一张图上进行横向对比
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False
    
    # 创建 1 行 2 列的子图画布 (左侧放 rfpp Cpk，右侧放 rfh1 Cpk)
    fig, axes = plt.subplots(1, 2, figsize=(18, 8), sharex=False)
    
    # 3.1 绘制 rfppwc_first 的 4 个分组的 Cpk 曲线在同一张图上
    sns.lineplot(
        ax=axes[0],
        x='date_str',
        y='cpk_rfpp',
        hue='group',
        style='group',
        data=daily_quality_df,
        markers=True,
        dashes=False,
        linewidth=2.5
    )
    axes[0].axhline(y=1.33, color='#2e7d32', linestyle='--', linewidth=1.8, label='黄金达标线 (Cpk=1.33)')
    axes[0].axhline(y=1.0, color='#c62828', linestyle='-', linewidth=1.8, label='能力警戒线 (Cpk=1.00)')
    axes[0].set_title("rfppwc_first 各组日度过程能力指数 (Cpk) 对比", fontsize=13, fontweight='bold')
    axes[0].set_xlabel("生产日期 (Date)", fontsize=11)
    axes[0].set_ylabel("Cpk 指数", fontsize=11)
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].legend(loc='lower left', frameon=True)
    
    # 3.2 绘制 rfh1wc_first 的 4 个分组的 Cpk 曲线在同一张图上
    sns.lineplot(
        ax=axes[1],
        x='date_str',
        y='cpk_rfh1',
        hue='group',
        style='group',
        data=daily_quality_df,
        markers=True,
        dashes=False,
        linewidth=2.5
    )
    axes[1].axhline(y=1.33, color='#2e7d32', linestyle='--', linewidth=1.8, label='黄金达标线 (Cpk=1.33)')
    axes[1].axhline(y=1.0, color='#c62828', linestyle='-', linewidth=1.8, label='能力警戒线 (Cpk=1.00)')
    axes[1].set_title("rfh1wc_first 各组日度过程能力指数 (Cpk) 对比", fontsize=13, fontweight='bold')
    axes[1].set_xlabel("生产日期 (Date)", fontsize=11)
    axes[1].set_ylabel("Cpk 指数", fontsize=11)
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].legend(loc='lower left', frameon=True)
    
    plt.suptitle("📊 各分组物理指标日度过程能力指数 (Cpk) 横向趋势对比追踪图", fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ 数据未加载，请运行前面的单元格。")


In [ ]:
if df is not None:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    # 1. 确保数值类型正确
    df['rfppwc_first'] = pd.to_numeric(df['rfppwc_first'], errors='coerce')
    df['rfh1wc_first'] = pd.to_numeric(df['rfh1wc_first'], errors='coerce')
    df['standard_rfpp'] = pd.to_numeric(df['standard_rfpp'], errors='coerce')
    df['standard_rfh1'] = pd.to_numeric(df['standard_rfh1'], errors='coerce')
    
    df['date_str'] = pd.to_datetime(df['ct_shiftdate']).dt.strftime('%Y-%m-%d')
    
    # 2. 分层级计算：先算 (date, group, article10) 的 Cpk，再按产量加权
    def calculate_weighted_daily_cpk(group_df):
        # 按日期、组和规格型号分组
        grouped = group_df.groupby(['date_str', 'group', 'article10'])
        article_stats = []
        
        for (date, grp, art), g_data in grouped:
            # 过滤掉单日单规格样本量太少的数据
            if len(g_data) < 5:
                continue
                
            std_rfpp = g_data['rfppwc_first'].std()
            std_rfh1 = g_data['rfh1wc_first'].std()
            
            # 排除无波动或无法计算标准差的规格
            if pd.isna(std_rfpp) or std_rfpp <= 1e-6 or pd.isna(std_rfh1) or std_rfh1 <= 1e-6:
                continue
                
            usl_rfpp = g_data['standard_rfpp'].dropna().iloc[0] if not g_data['standard_rfpp'].dropna().empty else None
            usl_rfh1 = g_data['standard_rfh1'].dropna().iloc[0] if not g_data['standard_rfh1'].dropna().empty else None
            
            if usl_rfpp is None or usl_rfh1 is None:
                continue
                
            mean_rfpp = g_data['rfppwc_first'].mean()
            mean_rfh1 = g_data['rfh1wc_first'].mean()
            
            # 计算单个规格型号在当天的 Cpk
            cpk_rfpp = (usl_rfpp - mean_rfpp) / (3 * std_rfpp)
            cpk_rfh1 = (usl_rfh1 - mean_rfh1) / (3 * std_rfh1)
            
            article_stats.append({
                'date_str': date,
                'group': grp,
                'article10': art,
                'sample_size': len(g_data),
                'cpk_rfpp': cpk_rfpp,
                'cpk_rfh1': cpk_rfh1
            })
            
        if len(article_stats) == 0:
            return pd.DataFrame()
            
        df_art = pd.DataFrame(article_stats)
        
        # 按日期和组别进行加权求和
        weighted_stats = []
        for (date, grp), g_data in df_art.groupby(['date_str', 'group']):
            # 针对 rfppwc_first 的有效 Cpk 进行加权
            valid_rfpp = g_data.dropna(subset=['cpk_rfpp'])
            if not valid_rfpp.empty:
                w_cpk_rfpp = (valid_rfpp['cpk_rfpp'] * valid_rfpp['sample_size']).sum() / valid_rfpp['sample_size'].sum()
            else:
                w_cpk_rfpp = np.nan
                
            # 针对 rfh1wc_first 的有效 Cpk 进行加权
            valid_rfh1 = g_data.dropna(subset=['cpk_rfh1'])
            if not valid_rfh1.empty:
                w_cpk_rfh1 = (valid_rfh1['cpk_rfh1'] * valid_rfh1['sample_size']).sum() / valid_rfh1['sample_size'].sum()
            else:
                w_cpk_rfh1 = np.nan
                
            weighted_stats.append({
                'date_str': date,
                'group': grp,
                'weighted_cpk_rfpp': w_cpk_rfpp,
                'weighted_cpk_rfh1': w_cpk_rfh1
            })
            
        return pd.DataFrame(weighted_stats)
        
    # 3. 执行计算
    daily_weighted_df = calculate_weighted_daily_cpk(df)
    daily_weighted_df = daily_weighted_df.sort_values(by=['group', 'date_str']).reset_index(drop=True)
    
    print("--- 基于规格加权后的每日过程能力指数 (Cpk) 明细表 (前 15 行) ---")
    display(daily_weighted_df.head(15).style.format({
        'weighted_cpk_rfpp': '{:.3f}',
        'weighted_cpk_rfh1': '{:.3f}'
    }))
    
    # 4. 可视化：在同一坐标系图表上进行多组折线对比
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    
    # 4.1 绘制加权后的 rfpp Cpk 折线图
    sns.lineplot(
        ax=axes[0],
        x='date_str',
        y='weighted_cpk_rfpp',
        hue='group',
        style='group',
        data=daily_weighted_df,
        markers=True,
        dashes=False,
        linewidth=2.5
    )
    axes[0].axhline(y=1.33, color='#2e7d32', linestyle='--', linewidth=1.8, label='黄金达标线 (Cpk=1.33)')
    axes[0].axhline(y=1.0, color='#c62828', linestyle='-', linewidth=1.8, label='能力警戒线 (Cpk=1.00)')
    axes[0].set_title("rfppwc_first 细分规格加权日度过程能力指数 (Cpk) 对比", fontsize=13, fontweight='bold')
    axes[0].set_xlabel("生产日期 (Date)", fontsize=11)
    axes[0].set_ylabel("加权 Cpk 指数", fontsize=11)
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].legend(loc='lower left', frameon=True)
    
    # 4.2 绘制加权后的 rfh1 Cpk 折线图
    sns.lineplot(
        ax=axes[1],
        x='date_str',
        y='weighted_cpk_rfh1',
        hue='group',
        style='group',
        data=daily_weighted_df,
        markers=True,
        dashes=False,
        linewidth=2.5
    )
    axes[1].axhline(y=1.33, color='#2e7d32', linestyle='--', linewidth=1.8, label='黄金达标线 (Cpk=1.33)')
    axes[1].axhline(y=1.0, color='#c62828', linestyle='-', linewidth=1.8, label='能力警戒线 (Cpk=1.00)')
    axes[1].set_title("rfh1wc_first 细分规格加权日度过程能力指数 (Cpk) 对比", fontsize=13, fontweight='bold')
    axes[1].set_xlabel("生产日期 (Date)", fontsize=11)
    axes[1].set_ylabel("加权 Cpk 指数", fontsize=11)
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].legend(loc='lower left', frameon=True)
    
    plt.suptitle("📊 各分组物理指标基于规格 (Article10) 产量加权后的日度 Cpk 横向趋势对比追踪图", fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ 数据未加载，请运行前面的单元格。")
